# Laya GPU server (for laya-vs-jev-pong)

Runs the same `laya` model this repo's local `server/laya_server.py` runs, but on a real GPU via Colab's free Tesla T4 - matching the ~40ms figure in Laya's own docs, instead of the ~1s+ this checkpoint takes on a typical CPU-only dev machine.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. (This menu is part of the Colab web UI at colab.research.google.com - opening this file in a local IDE like VS Code runs it against your own machine, not Colab's GPU, which defeats the point.)

You'll also need a free [ngrok](https://dashboard.ngrok.com/signup) account - Colab has no public IP of its own, so ngrok tunnels this notebook's local port 8787 out to a real HTTPS URL your browser can reach.

**Before running cell 3**, store your ngrok authtoken in Colab's secret manager rather than pasting it into a cell - a token typed directly into a cell gets saved as part of the notebook file, which is exactly how tokens end up committed to git history:

1. Grab your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
2. Click the key icon (🔑 **Secrets**) in the left sidebar of this Colab tab
3. Add a new secret named `NGROK_AUTHTOKEN`, paste the value, and toggle **Notebook access** on

The secret then lives in your own Colab account, not in this file - cell 3 reads it at runtime and never prints or stores the value anywhere.

In [ ]:
!pip install -q fastapi "uvicorn[standard]" laya pyngrok

import torch
assert torch.cuda.is_available(), "No GPU detected - check Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%%writefile colab_laya_server.py
# Same shape as this repo's server/laya_server.py, adapted to run
# standalone in a Colab VM: binds 0.0.0.0 (ngrok tunnels into it), forces
# device="cuda" instead of auto-detecting, and CORS defaults to the local
# Vite dev server's origin (override LAYA_CORS_ORIGIN if yours differs).
import os
import threading
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from typing import Any

import laya
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

CHECKPOINT = "convaiinnovations/laya"
CHECKPOINT_SUBFOLDER = "typed-decisions"
_agent = None
_predict_lock = threading.Lock()


@asynccontextmanager
async def lifespan(_: FastAPI) -> AsyncIterator[None]:
    global _agent
    _agent = laya.load(CHECKPOINT, subfolder=CHECKPOINT_SUBFOLDER, device="cuda")
    yield


app = FastAPI(title="laya-vs-jev-pong: Colab GPU server", lifespan=lifespan)

_allowed_origin = os.environ.get("LAYA_CORS_ORIGIN", "http://localhost:5173")
app.add_middleware(
    CORSMiddleware,
    allow_origins=[_allowed_origin],
    allow_methods=["GET", "POST"],
    allow_headers=["content-type", "ngrok-skip-browser-warning"],
)


class DecideRequest(BaseModel):
    state: dict[str, Any]
    questions: dict[str, Any]


@app.get("/health")
def health() -> dict[str, Any]:
    return {"status": "ok" if _agent is not None else "loading", "checkpoint": CHECKPOINT_SUBFOLDER}


@app.post("/decide")
def decide(req: DecideRequest) -> dict[str, Any]:
    if _agent is None:
        raise HTTPException(status_code=503, detail="model not loaded yet")
    if not _predict_lock.acquire(blocking=False):
        raise HTTPException(status_code=503, detail="a decide request is already in flight")
    try:
        result = _agent.predict(req.state, req.questions)
    finally:
        _predict_lock.release()
    return {"model": CHECKPOINT_SUBFOLDER, "answers": result["answers"]}

In [ ]:
import os

# If your local Vite dev server isn't on the default port, change this to match:
os.environ["LAYA_CORS_ORIGIN"] = "http://localhost:5173"

# Read the token from Colab's own secret manager (key icon, left sidebar) -
# never hardcode it in a cell. See the setup steps in the markdown cell above
# if you haven't added the NGROK_AUTHTOKEN secret yet.
from google.colab import userdata

try:
    NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
except userdata.SecretNotFoundError as e:
    raise RuntimeError(
        "No NGROK_AUTHTOKEN secret found. Click the key icon in the left sidebar, "
        "add a secret named NGROK_AUTHTOKEN with your token from "
        "https://dashboard.ngrok.com/get-started/your-authtoken, toggle notebook "
        "access on, then rerun this cell."
    ) from e
except userdata.NotebookAccessError as e:
    raise RuntimeError(
        "Found an NGROK_AUTHTOKEN secret but this notebook isn't allowed to read it - "
        "toggle 'Notebook access' on for it in the Secrets panel (key icon, left sidebar)."
    ) from e

from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)
del NGROK_AUTHTOKEN  # don't leave it sitting in a notebook variable longer than needed
print("ngrok authenticated.")

In [ ]:
import subprocess
import time

proc = subprocess.Popen(
    ["uvicorn", "colab_laya_server:app", "--host", "0.0.0.0", "--port", "8787"]
)
time.sleep(8)  # let the model finish loading onto the GPU before opening the tunnel

# ngrok.connect() returns an NgrokTunnel object, not a plain string - printing
# it directly (an earlier version of this cell did) prints its repr, e.g.
# NgrokTunnel: "https://..." -> "http://localhost:8787", which is NOT a valid
# value for VITE_LAYA_BASE_URL. .public_url is the actual URL string.
tunnel = ngrok.connect(8787, "http")
public_url = tunnel.public_url

print("Laya GPU server is live at:", public_url)
print()
print("Put this exact line in your local .env:")
print(f"VITE_LAYA_BASE_URL={public_url}")
print()
print("Then restart `npm run dev:web` (Vite only re-reads .env on startup).")
print("You no longer need the local server/laya_server.py running at all.")

## Notes

- **Keep this tab open.** The server (and the tunnel) only exist while this notebook cell is running. Closing the tab or letting Colab idle-disconnect kills it.
- **Free ngrok URLs change every time** you rerun the tunnel cell - update `VITE_LAYA_BASE_URL` again if you restart.
- **Colab free tier GPUs aren't guaranteed** and sessions can be reclaimed after a few hours of use; this is a demo/dev convenience, not something to depend on for a long-running session.
- **Never paste the raw authtoken into a cell.** Secrets stored via the key icon stay in your Colab account and are never saved into the .ipynb file itself, so this notebook is safe to keep in a public repo - a value typed directly into a cell, by contrast, gets saved with the file and would leak the moment you commit it.
- To stop the server cleanly: `proc.terminate()` in a new cell, then `ngrok.disconnect(public_url)`.